# BIRD Mini-Dev 500: Base vs SFT vs Targeted SFT Oracle Diagnosis

This notebook answers one specific question:

> Does each model fail because it cannot **generate** a correct SQL candidate, or because it cannot **select** the correct candidate as rank 1?

It evaluates three controlled conditions on the same official BIRD Mini-Dev 500 examples, always with BIRD evidence:

1. **Base + evidence**
2. **Original SFT + evidence**
3. **Targeted SFT + evidence**

For each question and model:

- Candidate 1 uses greedy decoding and defines **EX@1**.
- Candidates 2–5 use reproducible stochastic sampling.
- **Oracle@3/5** is correct if any candidate among the first 3/5 has the same execution result as the Gold SQL.
- Candidate diversity and valid-SQL rate are also recorded.

Only one model is kept in GPU memory at a time. Every condition is resumable and saves a CSV every five examples.

> Run the smoke test first with `DEBUG_LIMIT = 5`. After checking its rows, set `DEBUG_LIMIT = None` and rerun from the configuration cell. Smoke-test and full-run files are stored in separate subdirectories, so no deletion is required.


## 0. Controlled protocol

- Dataset: classic BIRD Mini-Dev 500, pinned to a fixed Hugging Face revision.
- Databases: official `minidev_0703.zip` SQLite package.
- Prompt: the same schema/evidence prompt used by the earlier four-way notebook.
- All three conditions receive the same question, schema, and real BIRD evidence.
- Execution match follows the official Mini-Dev evaluator semantics: `set(predicted_rows) == set(gold_rows)`.
- Decoding: one greedy candidate plus four sampled candidates (`temperature=0.7`, `top_p=0.95`).
- The notebook never trains on Mini-Dev500; it only diagnoses existing models.

The full run performs 1,500 prompts and evaluates 7,500 SQL candidates. An A100 is recommended.


## 1. Install dependencies


In [20]:
!pip -q install -U \
    "transformers>=4.48" \
    "peft>=0.14" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "datasets>=3.2" \
    "gdown>=5.2" \
    "sqlglot>=26.0" \
    pandas matplotlib seaborn tqdm

## 2. Imports, random seeds, and Google Drive


In [21]:
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import sqlite3
import statistics
import time
import warnings
import zipfile
from pathlib import Path
from urllib.parse import quote

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sqlglot
import torch
from datasets import load_dataset
from IPython.display import display
from peft import PeftConfig, PeftModel
from scipy.stats import binomtest
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU was detected. In Colab, select Runtime → Change runtime type → GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())

from google.colab import drive
drive.mount("/content/drive")


GPU: NVIDIA A100-SXM4-40GB
CUDA: 12.8
BF16 supported: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Configuration

You only need to confirm the two adapter paths below. The original SFT path is prefilled from the previous project layout. Set `TARGETED_SFT_ADAPTER_DIR` to the `final_adapter` directory produced by the targeted training notebook.

The result directory is new, so it will not overwrite earlier four-way or single-output results.


In [22]:
# ========================= PROJECT LAYOUT =========================
PROJECT_ROOT = Path("/content/drive/MyDrive/bird-text2sql-sft")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

ORIGINAL_SFT_ADAPTER_DIR = (
    PROJECT_ROOT
    / "training"
    / "qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192"
    / "final_adapter"
)

# IMPORTANT: replace this folder name if your targeted run used another name.
TARGETED_SFT_ADAPTER_DIR = (
    PROJECT_ROOT
    / "training"
    / "qwen2.5-coder-7b-bf16-lora-targeted-sft-v1-lr2e-5-len8192"
    / "final_adapter"
)

PROJECT_DIR = RESULTS_DIR / "minidev500_base_sft_targeted_oracle_diagnosis"

for directory in [DATA_DIR, PROCESSED_DIR, RESULTS_DIR, PROJECT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
# =================================================================

BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Official Mini-Dev 500 dataset and database package.
HF_DATASET_ID = "birdsql/bird_mini_dev"
HF_REVISION = "f65faf4ae3b638c1fa6df1d3370c8d92c8366301"
HF_SPLIT = "mini_dev_sqlite"
OFFICIAL_DB_GOOGLE_DRIVE_ID = "13VLWIwpw5E3d5DUkMvzw7hvHE67a4XkG"

LOCAL_DATA_ROOT = DATA_DIR / "bird_minidev500"
LOCAL_ZIP_PATH = LOCAL_DATA_ROOT / "minidev_0703.zip"
LOCAL_EXTRACT_DIR = LOCAL_DATA_ROOT / "extracted"
HF_CACHE_DIR = DATA_DIR / "hf_cache"

# Model and generation settings.
LOAD_IN_4BIT = False       # BF16 is preferred on an A100; set True only if memory requires it.
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 256
SQL_TIMEOUT_SECONDS = 30.0

NUM_CANDIDATES = 5
ORACLE_K_VALUES = (1, 3, 5)
NUM_SAMPLED_CANDIDATES = NUM_CANDIDATES - 1
SAMPLING_TEMPERATURE = 0.7
SAMPLING_TOP_P = 0.95

assert NUM_CANDIDATES == max(ORACLE_K_VALUES)

# Run controls.
RESUME = True
FORCE_RESTART = False
SAVE_EVERY = 5
DEBUG_LIMIT = None          # First smoke test: 5. Full run: None.

# Smoke and full outputs are isolated; changing DEBUG_LIMIT never mixes their files.
RUN_DIR = PROJECT_DIR / (
    f"smoke_n{int(DEBUG_LIMIT)}" if DEBUG_LIMIT is not None else "full_n500"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_VERSION = "bird_schema_with_evidence_chat_v2"

EXPERIMENTS = {
    "base_with_evidence": {
        "display_name": "Base + evidence",
        "model_type": "base",
        "adapter_dir": None,
    },
    "original_sft_with_evidence": {
        "display_name": "Original SFT + evidence",
        "model_type": "original_sft",
        "adapter_dir": ORIGINAL_SFT_ADAPTER_DIR,
    },
    "targeted_sft_with_evidence": {
        "display_name": "Targeted SFT + evidence",
        "model_type": "targeted_sft",
        "adapter_dir": TARGETED_SFT_ADAPTER_DIR,
    },
}

print("Results:", RUN_DIR)
for label, spec in EXPERIMENTS.items():
    print(f"{label}: {spec['adapter_dir'] or BASE_MODEL_ID}")


Results: /content/drive/MyDrive/bird-text2sql-sft/results/minidev500_base_sft_targeted_oracle_diagnosis/full_n500
base_with_evidence: Qwen/Qwen2.5-Coder-7B-Instruct
original_sft_with_evidence: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/final_adapter
targeted_sft_with_evidence: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-targeted-sft-v1-lr2e-5-len8192/final_adapter


### 3.1 Validate both adapter directories


In [23]:
def valid_adapter_dir(path):
    path = Path(path)
    has_weights = (
        (path / "adapter_model.safetensors").exists()
        or (path / "adapter_model.bin").exists()
    )
    return (path / "adapter_config.json").exists() and has_weights


def find_adapter_candidates(training_root, max_results=50):
    candidates = []
    if not training_root.exists():
        return candidates
    for config_path in training_root.rglob("adapter_config.json"):
        folder = config_path.parent
        if valid_adapter_dir(folder):
            candidates.append(folder)
            if len(candidates) >= max_results:
                break
    return candidates


invalid = []
for label, spec in EXPERIMENTS.items():
    adapter_dir = spec["adapter_dir"]
    if adapter_dir is None:
        continue
    if not valid_adapter_dir(adapter_dir):
        invalid.append((label, adapter_dir))

if invalid:
    print("Invalid configured adapter path(s):")
    for label, path in invalid:
        print(" -", label, ":", path)
    print("\nAvailable adapter directories under training/:")
    candidates = find_adapter_candidates(PROJECT_ROOT / "training")
    for candidate in candidates:
        print(" -", candidate)
    raise FileNotFoundError(
        "Update ORIGINAL_SFT_ADAPTER_DIR and/or TARGETED_SFT_ADAPTER_DIR "
        "in the configuration cell, then rerun this cell."
    )

for label, spec in EXPERIMENTS.items():
    adapter_dir = spec["adapter_dir"]
    if adapter_dir is None:
        continue
    peft_config = PeftConfig.from_pretrained(str(adapter_dir))
    print(label)
    print("  adapter:", adapter_dir)
    print("  recorded base:", peft_config.base_model_name_or_path)


original_sft_with_evidence
  adapter: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/final_adapter
  recorded base: Qwen/Qwen2.5-Coder-7B-Instruct
targeted_sft_with_evidence
  adapter: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-targeted-sft-v1-lr2e-5-len8192/final_adapter
  recorded base: Qwen/Qwen2.5-Coder-7B-Instruct


## 4. Load and pin the official Mini-Dev 500 dataset


In [24]:
dataset_dict = load_dataset(
    HF_DATASET_ID,
    revision=HF_REVISION,
    cache_dir=str(HF_CACHE_DIR),
)

if HF_SPLIT not in dataset_dict:
    raise KeyError(f"Could not find split={HF_SPLIT}; available splits: {list(dataset_dict.keys())}")

raw_split = dataset_dict[HF_SPLIT]
print(raw_split)
print("Columns:", raw_split.column_names)


def first_present(example, names, default=None):
    for name in names:
        if name in example and example[name] is not None:
            return example[name]
    return default


records = []
for position, example in enumerate(raw_split):
    gold_sql = first_present(example, ["SQL", "sql", "query", "gold_sql"])
    if gold_sql is None:
        raise KeyError(f"Gold SQL was not found for example {position}. Fields={list(example.keys())}")

    records.append({
        "position": position,
        "source_index": first_present(example, ["index", "idx", "id"], position),
        "db_id": str(first_present(example, ["db_id", "database_id"])),
        "question": str(first_present(example, ["question", "utterance"])),
        "evidence": str(first_present(example, ["evidence", "external_knowledge"], "") or ""),
        "gold_sql": str(gold_sql).strip(),
        "difficulty": str(first_present(example, ["difficulty", "level"], "unknown")).lower(),
    })

eval_df = pd.DataFrame(records)

assert len(eval_df) == 500, f"Expected 500 examples, found {len(eval_df)}"
assert eval_df["db_id"].nunique() == 11, (
    f"Expected 11 databases, found {eval_df['db_id'].nunique()}"
)
assert eval_df["question"].notna().all()
assert eval_df["gold_sql"].str.len().gt(0).all()

dataset_payload = eval_df.to_json(orient="records", force_ascii=False)
DATASET_SHA256 = hashlib.sha256(dataset_payload.encode("utf-8")).hexdigest()

# Store the pinned, normalized 500-example evaluation snapshot under processed/.
snapshot_path = PROCESSED_DIR / "mini_dev_500_snapshot.jsonl"
eval_df.to_json(
    snapshot_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Rows:", len(eval_df))
print("Databases:", eval_df["db_id"].nunique())
print("Dataset SHA256:", DATASET_SHA256)
print("Processed snapshot:", snapshot_path)
display(eval_df.head(3))
display(eval_df["difficulty"].value_counts(dropna=False).rename("count").to_frame())
display(eval_df["db_id"].value_counts().sort_index().rename("count").to_frame())


Dataset({
    features: ['question_id', 'db_id', 'question', 'evidence', 'SQL', 'difficulty'],
    num_rows: 500
})
Columns: ['question_id', 'db_id', 'question', 'evidence', 'SQL', 'difficulty']
Rows: 500
Databases: 11
Dataset SHA256: c38826f6825c30bd2d3916f30dab8f259e149bf9ca8d2d0bb454d8493105b52c
Processed snapshot: /content/drive/MyDrive/bird-text2sql-sft/processed/mini_dev_500_snapshot.jsonl


,position,source_index,db_id,question,evidence,gold_sql,difficulty
0,0,0,debit_card_specializing,What is the ratio of customers who pay in EUR against customers who pay in CZK?,ratio of customers who pay in EUR against customers who pay in CZK = count(Currency = 'EUR') / count(Currency = 'CZK').,"SELECT CAST(SUM(IIF(Currency = 'EUR', 1, 0)) AS FLOAT) / SUM(IIF(Currency = 'CZK', 1, 0)) AS ratio FROM customers",simple
1,1,1,debit_card_specializing,"In 2012, who had the least consumption in LAM?",Year 2012 can be presented as Between 201201 And 201212; The first 4 strings of the Date values in the yearmonth table can represent year.,"SELECT T1.CustomerID FROM customers AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T1.Segment = 'LAM' AND SUBSTR(T2.Date, 1, 4) = '...",moderate
2,2,2,debit_card_specializing,What was the average monthly consumption of customers in SME for the year 2013?,Average Monthly consumption = AVG(Consumption) / 12; Year 2013 can be presented as Between 201301 And 201312; The first 4 strings of the Date values in the ...,"SELECT AVG(T2.Consumption) / 12 FROM customers AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE SUBSTR(T2.Date, 1, 4) = '2013' AND T1...",moderate


,count
difficulty,
moderate,250
simple,148
challenging,102


,count
db_id,
california_schools,30
card_games,52
codebase_community,49
debit_card_specializing,30
european_football_2,51
financial,32
formula_1,66
student_club,48
superhero,52


## 5. Download and extract the official SQLite databases


In [25]:
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not LOCAL_ZIP_PATH.exists():
    print("Downloading the database package from the official BIRD Google Drive...")
    downloaded = gdown.download(
        id=OFFICIAL_DB_GOOGLE_DRIVE_ID,
        output=str(LOCAL_ZIP_PATH),
        quiet=False,
    )
    if downloaded is None or not LOCAL_ZIP_PATH.exists():
        raise RuntimeError("The database package download failed. Rerun this cell.")
else:
    print("The archive already exists; skipping download:", LOCAL_ZIP_PATH)

if not zipfile.is_zipfile(LOCAL_ZIP_PATH):
    raise RuntimeError(f"The downloaded file is not a valid ZIP archive: {LOCAL_ZIP_PATH}")

extract_marker = LOCAL_EXTRACT_DIR / ".extraction_complete"
if not extract_marker.exists():
    if LOCAL_EXTRACT_DIR.exists():
        shutil.rmtree(LOCAL_EXTRACT_DIR)
    LOCAL_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting the archive...")
    with zipfile.ZipFile(LOCAL_ZIP_PATH) as zf:
        zf.extractall(LOCAL_EXTRACT_DIR)
    extract_marker.touch()
    print("Extraction completed:", LOCAL_EXTRACT_DIR)
else:
    print("The archive is already extracted; skipping:", LOCAL_EXTRACT_DIR)


The archive already exists; skipping download: /content/drive/MyDrive/bird-text2sql-sft/data/bird_minidev500/minidev_0703.zip
The archive is already extracted; skipping: /content/drive/MyDrive/bird-text2sql-sft/data/bird_minidev500/extracted


## 6. Locate the 11 databases and copy them to the Colab SSD


In [26]:
import shutil
import sqlite3
from pathlib import Path
from urllib.parse import quote

# Requirements:
# 1. eval_df has already been created.
# 2. LOCAL_EXTRACT_DIR points to the extracted Mini-Dev database directory.

assert "eval_df" in globals(), "Run the Mini-Dev dataset loading cell first."
assert "LOCAL_EXTRACT_DIR" in globals(), (
    "Run the database download and extraction cell first."
)
assert LOCAL_EXTRACT_DIR.exists(), (
    f"Database directory does not exist: {LOCAL_EXTRACT_DIR}"
)


def collect_sqlite_files(root: Path):
    """Find SQLite database files under the extracted directory."""
    suffixes = {".sqlite", ".sqlite3", ".db"}

    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in suffixes
    )


def resolve_db_path(db_id: str, candidates):
    """Find the SQLite file corresponding to one BIRD db_id."""

    # Preferred structure: .../<db_id>/<db_id>.sqlite
    preferred = [
        path
        for path in candidates
        if path.stem == db_id and path.parent.name == db_id
    ]
    if preferred:
        return min(preferred, key=lambda path: len(str(path)))

    # Match the filename, such as california_schools.sqlite
    stem_matches = [
        path for path in candidates
        if path.stem == db_id
    ]
    if stem_matches:
        return min(stem_matches, key=lambda path: len(str(path)))

    # Match the parent directory.
    parent_matches = [
        path for path in candidates
        if path.parent.name == db_id
    ]
    if parent_matches:
        return min(parent_matches, key=lambda path: len(str(path)))

    raise FileNotFoundError(
        f"Could not find a SQLite file for db_id={db_id}"
    )


# Step 1: Locate the source databases.
sqlite_files = collect_sqlite_files(LOCAL_EXTRACT_DIR)

print("SQLite-like files found:", len(sqlite_files))

if not sqlite_files:
    raise FileNotFoundError(
        f"No SQLite files were found under {LOCAL_EXTRACT_DIR}"
    )

db_ids = sorted(eval_df["db_id"].dropna().unique())

print("Unique database IDs in Mini-Dev:", len(db_ids))
assert len(db_ids) == 11, (
    f"Expected 11 Mini-Dev databases, but found {len(db_ids)}"
)

SOURCE_DB_PATHS = {
    db_id: resolve_db_path(db_id, sqlite_files)
    for db_id in db_ids
}

print("All source databases were resolved.")


# Step 2: Copy the 11 databases to the local Colab SSD.
LOCAL_DB_CACHE = Path("/content/bird_minidev500_sqlite")
LOCAL_DB_CACHE.mkdir(parents=True, exist_ok=True)

DB_PATHS = {}

for position, db_id in enumerate(db_ids, start=1):
    source_path = SOURCE_DB_PATHS[db_id]

    target_dir = LOCAL_DB_CACHE / db_id
    target_dir.mkdir(parents=True, exist_ok=True)

    target_path = target_dir / source_path.name

    source_size = source_path.stat().st_size
    target_is_complete = (
        target_path.exists()
        and target_path.stat().st_size == source_size
    )

    if target_is_complete:
        status = "cached"
    else:
        shutil.copy2(source_path, target_path)
        status = "copied"

    DB_PATHS[db_id] = target_path

    print(
        f"[{position:02d}/{len(db_ids)}] "
        f"{db_id}: {status} "
        f"({source_size / 1024**2:.2f} MB)"
    )


# Step 3: Perform only a fast database-opening check.
health_rows = []

for db_id, db_path in DB_PATHS.items():
    uri = f"file:{quote(str(db_path.resolve()))}?mode=ro"

    with sqlite3.connect(uri, uri=True, timeout=10) as connection:
        table_count = connection.execute(
            """
            SELECT COUNT(*)
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            """
        ).fetchone()[0]

    health_rows.append(
        {
            "db_id": db_id,
            "db_path": str(db_path),
            "storage": "Colab local SSD",
            "table_count": table_count,
            "size_mb": round(db_path.stat().st_size / 1024**2, 2),
        }
    )

    print(f"Opened {db_id}: {table_count} tables")


db_health_df = pd.DataFrame(health_rows)

assert len(DB_PATHS) == 11
assert all(path.exists() for path in DB_PATHS.values())
assert db_health_df["table_count"].gt(0).all(), db_health_df

display(db_health_df)

print("DB_PATHS has been created successfully.")
print("All evaluation queries will use the local Colab SSD.")


SQLite-like files found: 22
Unique database IDs in Mini-Dev: 11
All source databases were resolved.
[01/11] california_schools: cached (10.60 MB)
[02/11] card_games: cached (249.69 MB)
[03/11] codebase_community: cached (459.12 MB)
[04/11] debit_card_specializing: cached (33.03 MB)
[05/11] european_football_2: cached (570.06 MB)
[06/11] financial: cached (67.99 MB)
[07/11] formula_1: cached (21.32 MB)
[08/11] student_club: cached (2.52 MB)
[09/11] superhero: cached (0.23 MB)
[10/11] thrombosis_prediction: cached (6.99 MB)
[11/11] toxicology: cached (2.55 MB)
Opened california_schools: 3 tables
Opened card_games: 6 tables
Opened codebase_community: 8 tables
Opened debit_card_specializing: 5 tables
Opened european_football_2: 7 tables
Opened financial: 8 tables
Opened formula_1: 13 tables
Opened student_club: 8 tables
Opened superhero: 10 tables
Opened thrombosis_prediction: 3 tables
Opened toxicology: 4 tables


,db_id,db_path,storage,table_count,size_mb
0,california_schools,/content/bird_minidev500_sqlite/california_schools/california_schools.sqlite,Colab local SSD,3,10.60
1,card_games,/content/bird_minidev500_sqlite/card_games/card_games.sqlite,Colab local SSD,6,249.69
2,codebase_community,/content/bird_minidev500_sqlite/codebase_community/codebase_community.sqlite,Colab local SSD,8,459.12
3,debit_card_specializing,/content/bird_minidev500_sqlite/debit_card_specializing/debit_card_specializing.sqlite,Colab local SSD,5,33.03
4,european_football_2,/content/bird_minidev500_sqlite/european_football_2/european_football_2.sqlite,Colab local SSD,7,570.06
5,financial,/content/bird_minidev500_sqlite/financial/financial.sqlite,Colab local SSD,8,67.99
6,formula_1,/content/bird_minidev500_sqlite/formula_1/formula_1.sqlite,Colab local SSD,13,21.32
7,student_club,/content/bird_minidev500_sqlite/student_club/student_club.sqlite,Colab local SSD,8,2.52
8,superhero,/content/bird_minidev500_sqlite/superhero/superhero.sqlite,Colab local SSD,10,0.23
9,thrombosis_prediction,/content/bird_minidev500_sqlite/thrombosis_prediction/thrombosis_prediction.sqlite,Colab local SSD,3,6.99


DB_PATHS has been created successfully.
All evaluation queries will use the local Colab SSD.


## 7. Build the schema + evidence prompt

All three models receive real BIRD evidence. Do not edit this prompt after starting the full run; the run fingerprint prevents incompatible partial files from being mixed.


In [27]:
SCHEMA_CACHE = {}


def get_sqlite_schema(db_path):
    cache_key = str(Path(db_path).resolve())
    if cache_key in SCHEMA_CACHE:
        return SCHEMA_CACHE[cache_key]

    uri = f"file:{quote(cache_key)}?mode=ro"
    with sqlite3.connect(uri, uri=True) as conn:
        rows = conn.execute(
            """
            SELECT name, sql
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
              AND sql IS NOT NULL
            ORDER BY name
            """
        ).fetchall()

    schema = "\n\n".join(sql.strip() + ";" for _, sql in rows)
    SCHEMA_CACHE[cache_key] = schema
    return schema


SYSTEM_PROMPT = (
    "You are an expert SQLite developer. "
    "Given a database schema, a question, and optional evidence, "
    "write one valid SQLite query that answers the question. "
    "Return only the SQL query without explanations or Markdown."
)


def build_messages(record):
    schema = get_sqlite_schema(DB_PATHS[record["db_id"]])
    evidence = str(record.get("evidence", "") or "").strip() or "None"

    user_prompt = f"""Database ID:
{record['db_id']}

SQLite schema:
{schema}

External knowledge:
{evidence}

Question:
{record['question']}

Write exactly one executable SQLite query that answers the question. Return SQL only."""

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


preview_record = eval_df.iloc[0].to_dict()
print(json.dumps(build_messages(preview_record), ensure_ascii=False, indent=2))
evidence_coverage = eval_df["evidence"].fillna("").str.strip().ne("").mean()
print(f"\nNon-empty BIRD evidence: {evidence_coverage:.1%}")


[
  {
    "role": "system",
    "content": "You are an expert SQLite developer. Given a database schema, a question, and optional evidence, write one valid SQLite query that answers the question. Return only the SQL query without explanations or Markdown."
  },
  {
    "role": "user",
    "content": "Database ID:\ndebit_card_specializing\n\nSQLite schema:\nCREATE TABLE customers\n(\n    CustomerID INTEGER UNIQUE     not null\n        primary key,\n    Segment    TEXT null,\n    Currency   TEXT null\n);\n\nCREATE TABLE gasstations\n(\n    GasStationID INTEGER    UNIQUE   not null\n        primary key,\n    ChainID      INTEGER          null,\n    Country      TEXT null,\n    Segment      TEXT null\n);\n\nCREATE TABLE products\n(\n    ProductID   INTEGER   UNIQUE      not null\n        primary key,\n    Description TEXT null\n);\n\nCREATE TABLE \"transactions_1k\"\n(\n    TransactionID INTEGER\n        primary key autoincrement,\n    Date          DATE,\n    Time          TEXT,\n    Cust

## 8. SQL extraction and official execution matching


In [28]:
def first_sql_statement(text):
    '''Keep the first SQL statement and ignore semicolons inside quoted strings.'''
    quote_char = None
    i = 0
    while i < len(text):
        ch = text[i]
        if quote_char is None:
            if ch in {"'", '"', '`'}:
                quote_char = ch
            elif ch == ";":
                return text[: i + 1]
        else:
            if ch == quote_char:
                # Two consecutive quote characters in SQL represent an escaped quote
                if i + 1 < len(text) and text[i + 1] == quote_char:
                    i += 1
                else:
                    quote_char = None
        i += 1
    return text


def extract_sql(raw_text):
    text = (raw_text or "").strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.IGNORECASE | re.DOTALL).strip()

    fenced = re.search(
        r"```(?:sql|sqlite)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if fenced:
        text = fenced.group(1).strip()

    start = re.search(r"\b(?:SELECT|WITH)\b", text, flags=re.IGNORECASE)
    if start:
        text = text[start.start():]

    text = first_sql_statement(text).strip()
    text = re.sub(r"\s*(?:<\|im_end\|>|<\|endoftext\|>)\s*$", "", text).strip()
    return text


def normalize_sql(sql):
    if not sql or not sql.strip():
        return ""
    try:
        parsed = sqlglot.parse_one(sql, read="sqlite")
        return parsed.sql(dialect="sqlite", pretty=False, normalize=True).strip().lower()
    except Exception:
        fallback = re.sub(r"\s+", " ", sql.strip().rstrip(";"))
        return fallback.lower()


def execute_read_only(db_path, sql, timeout_seconds=30.0):
    '''Execute a query through a read-only SQLite connection with a progress-handler timeout.'''
    if not sql or not sql.strip():
        return None, 0.0, "empty SQL"

    start_time = time.perf_counter()
    deadline = start_time + timeout_seconds
    uri = f"file:{quote(str(Path(db_path).resolve()))}?mode=ro"
    conn = None
    try:
        conn = sqlite3.connect(uri, uri=True, timeout=timeout_seconds)
        conn.execute("PRAGMA query_only = ON")
        conn.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10_000)
        rows = conn.execute(sql).fetchall()
        elapsed = time.perf_counter() - start_time
        return rows, elapsed, ""
    except Exception as exc:
        elapsed = time.perf_counter() - start_time
        message = str(exc)
        if "interrupted" in message.lower() and time.perf_counter() >= deadline:
            message = f"timeout after {timeout_seconds:.1f}s"
        return None, elapsed, message
    finally:
        if conn is not None:
            conn.close()


def official_execution_equal(predicted_rows, gold_rows):
    '''Match the official BIRD Mini-Dev evaluation_ex.py semantics: set(pred) == set(gold).'''
    if predicted_rows is None or gold_rows is None:
        return False
    return set(predicted_rows) == set(gold_rows)


# Run a Gold SQL sanity check to catch an incorrect database package or mapping.
gold_check_rows = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Gold SQL sanity check"):
    result, elapsed, error = execute_read_only(
        DB_PATHS[row["db_id"]],
        row["gold_sql"],
        SQL_TIMEOUT_SECONDS,
    )
    gold_check_rows.append({
        "position": int(row["position"]),
        "gold_valid": result is not None,
        "gold_execution_seconds": elapsed,
        "gold_error": error,
    })

gold_check_df = pd.DataFrame(gold_check_rows)
display(gold_check_df["gold_valid"].value_counts(dropna=False).rename("count").to_frame())

if not gold_check_df["gold_valid"].all():
    display(gold_check_df.loc[~gold_check_df["gold_valid"]].head(20))
    raise RuntimeError(
        "At least one Gold SQL query could not be executed. Check the database package, db_id mapping, and SQLite version."
    )


Gold SQL sanity check:   0%|          | 0/500 [00:00<?, ?it/s]

,count
gold_valid,
True,500


## 9. Load one model at a time and generate five candidates

The base weights are reloaded for each adapter, but the preceding model is deleted before the next one is loaded. This prevents the Base, original SFT, and targeted SFT models from occupying GPU memory simultaneously.


In [29]:
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def make_quantization_config():
    if not LOAD_IN_4BIT:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )


def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        cache_dir=str(HF_CACHE_DIR),
        use_fast=True,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    return tokenizer


def load_eval_model(adapter_dir=None):
    kwargs = dict(
        cache_dir=str(HF_CACHE_DIR),
        device_map="auto",
        torch_dtype=COMPUTE_DTYPE,
        low_cpu_mem_usage=True,
    )
    quant_config = make_quantization_config()
    if quant_config is not None:
        kwargs["quantization_config"] = quant_config

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **kwargs)
    if adapter_dir is None:
        model = base_model
    else:
        model = PeftModel.from_pretrained(
            base_model,
            str(adapter_dir),
            is_trainable=False,
        )
    model.eval()
    return model


def model_input_device(model):
    return model.get_input_embeddings().weight.device


def generated_token_count(token_ids, eos_token_id):
    if eos_token_id is None:
        return int(token_ids.numel())
    eos_positions = (token_ids == eos_token_id).nonzero(as_tuple=False)
    if len(eos_positions):
        return int(eos_positions[0].item() + 1)
    return int(token_ids.numel())


def decode_candidate_batch(output_ids, input_token_count, tokenizer, rank_offset):
    candidates = []
    for batch_index, sequence_ids in enumerate(output_ids):
        new_ids = sequence_ids[input_token_count:]
        raw_generation = tokenizer.decode(new_ids, skip_special_tokens=True)
        candidates.append({
            "rank": int(rank_offset + batch_index),
            "raw_generation": raw_generation,
            "predicted_sql": extract_sql(raw_generation),
            "output_token_count": generated_token_count(
                new_ids, tokenizer.eos_token_id
            ),
        })
    return candidates


@torch.inference_mode()
def generate_candidates(model, tokenizer, record):
    messages = build_messages(record)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=False,
    )
    input_token_count = int(inputs["input_ids"].shape[1])
    device = model_input_device(model)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    common_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()

    greedy_ids = model.generate(
        **inputs,
        **common_kwargs,
        do_sample=False,
        num_return_sequences=1,
    )
    candidates = decode_candidate_batch(
        greedy_ids, input_token_count, tokenizer, rank_offset=1
    )

    sample_seed = int(SEED + int(record["position"]))
    torch.manual_seed(sample_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(sample_seed)

    sampled_ids = model.generate(
        **inputs,
        **common_kwargs,
        do_sample=True,
        temperature=SAMPLING_TEMPERATURE,
        top_p=SAMPLING_TOP_P,
        num_return_sequences=NUM_SAMPLED_CANDIDATES,
    )
    candidates.extend(
        decode_candidate_batch(
            sampled_ids, input_token_count, tokenizer, rank_offset=2
        )
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    if len(candidates) != NUM_CANDIDATES:
        raise RuntimeError(
            f"Expected {NUM_CANDIDATES} candidates, generated {len(candidates)}"
        )

    return {
        "candidates": candidates,
        "input_token_count": input_token_count,
        "total_output_token_count": int(sum(
            candidate["output_token_count"] for candidate in candidates
        )),
        "generation_seconds": time.perf_counter() - started,
        "sample_seed": sample_seed,
    }


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    time.sleep(1)
    if torch.cuda.is_available():
        print(
            "GPU memory allocated (GB):",
            round(torch.cuda.memory_allocated() / 1024**3, 3),
        )


## 10. Resumable evaluation

Each row stores all five SQL candidates, per-candidate execution outcomes, EX@1, Oracle@3, Oracle@5, first correct rank, valid rate, unique-candidate count, token counts, and timing.


In [30]:
def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    raise TypeError(type(value).__name__)


def adapter_signature(adapter_dir):
    if adapter_dir is None:
        return None
    adapter_dir = Path(adapter_dir)
    files = []
    for name in ["adapter_config.json", "adapter_model.safetensors", "adapter_model.bin"]:
        path = adapter_dir / name
        if path.exists():
            item = {"name": name, "size": path.stat().st_size}
            if name.endswith(".json"):
                item["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
            files.append(item)
    return {"path": str(adapter_dir), "files": files}


def run_config(experiment_label, spec):
    payload = {
        "experiment_label": experiment_label,
        "display_name": spec["display_name"],
        "model_type": spec["model_type"],
        "base_model_id": BASE_MODEL_ID,
        "adapter": adapter_signature(spec["adapter_dir"]),
        "dataset_id": HF_DATASET_ID,
        "dataset_revision": HF_REVISION,
        "dataset_split": HF_SPLIT,
        "dataset_sha256": DATASET_SHA256,
        "prompt_version": PROMPT_VERSION,
        "system_prompt": SYSTEM_PROMPT,
        "use_evidence": True,
        "load_in_4bit": LOAD_IN_4BIT,
        "compute_dtype": str(COMPUTE_DTYPE),
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_candidates": NUM_CANDIDATES,
        "oracle_k_values": list(ORACLE_K_VALUES),
        "candidate_1_decoding": "greedy",
        "sampled_candidate_count": NUM_SAMPLED_CANDIDATES,
        "sampled_candidates_decoding": "multinomial_sampling",
        "sampling_temperature": SAMPLING_TEMPERATURE,
        "sampling_top_p": SAMPLING_TOP_P,
        "sample_seed_rule": "SEED + position",
        "seed": SEED,
        "sql_timeout_seconds": SQL_TIMEOUT_SECONDS,
        "debug_limit": DEBUG_LIMIT,
        "official_ex_semantics": "set(predicted_rows) == set(gold_rows)",
    }
    canonical = json.dumps(
        payload, sort_keys=True, ensure_ascii=False, default=json_default
    )
    payload["fingerprint"] = hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()
    return payload


def atomic_write_csv(df, path):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def atomic_write_json(obj, path):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, default=json_default),
        encoding="utf-8",
    )
    os.replace(temporary_path, path)


def bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes"})


def evaluate_candidate_set(record, generation, experiment_label, spec):
    db_path = DB_PATHS[record["db_id"]]
    gold_sql = record["gold_sql"]
    gold_rows, gold_exec_seconds, gold_error = execute_read_only(
        db_path, gold_sql, SQL_TIMEOUT_SECONDS
    )

    evaluated = []
    flat_fields = {}

    for candidate in generation["candidates"]:
        rank = int(candidate["rank"])
        predicted_sql = candidate["predicted_sql"]
        predicted_rows, exec_seconds, sql_error = execute_read_only(
            db_path, predicted_sql, SQL_TIMEOUT_SECONDS
        )
        item = {
            **candidate,
            "valid_sql": predicted_rows is not None,
            "execution_match": official_execution_equal(predicted_rows, gold_rows),
            "normalized_exact_match": (
                normalize_sql(predicted_sql) == normalize_sql(gold_sql)
            ),
            "predicted_execution_seconds": exec_seconds,
            "sql_error": sql_error,
        }
        evaluated.append(item)

        prefix = f"candidate_{rank}"
        for field in [
            "raw_generation",
            "predicted_sql",
            "output_token_count",
            "valid_sql",
            "execution_match",
            "normalized_exact_match",
            "predicted_execution_seconds",
            "sql_error",
        ]:
            flat_fields[f"{prefix}_{field}"] = item[field]

    matches = [bool(item["execution_match"]) for item in evaluated]
    first_correct_rank = next(
        (item["rank"] for item in evaluated if item["execution_match"]),
        np.nan,
    )
    normalized_candidates = [
        normalize_sql(item["predicted_sql"]) for item in evaluated
    ]
    unique_candidate_count = len(set(normalized_candidates))
    first = evaluated[0]

    result = {
        "experiment": experiment_label,
        "display_name": spec["display_name"],
        "model_type": spec["model_type"],
        "position": int(record["position"]),
        "source_index": record["source_index"],
        "db_id": record["db_id"],
        "db_path": str(db_path),
        "difficulty": record["difficulty"],
        "question": record["question"],
        "evidence": record["evidence"],
        "gold_sql": gold_sql,
        "gold_valid": gold_rows is not None,
        "gold_execution_seconds": gold_exec_seconds,
        "gold_sql_error": gold_error,
        "candidate_count": NUM_CANDIDATES,
        "unique_candidate_count": unique_candidate_count,
        "candidate_diversity_rate": unique_candidate_count / NUM_CANDIDATES,
        "sample_seed": generation["sample_seed"],
        "ex_at_1": bool(matches[0]),
        "oracle_at_3": bool(any(matches[:3])),
        "oracle_at_5": bool(any(matches[:5])),
        "oracle_hit_rank": first_correct_rank,
        "candidate_valid_rate": float(np.mean([
            item["valid_sql"] for item in evaluated
        ])),
        "input_token_count": generation["input_token_count"],
        "output_token_count": first["output_token_count"],
        "total_output_token_count": generation["total_output_token_count"],
        "generation_seconds": generation["generation_seconds"],
        "valid_sql": bool(first["valid_sql"]),
        "execution_match": bool(first["execution_match"]),
        "normalized_exact_match": bool(first["normalized_exact_match"]),
        "predicted_sql": first["predicted_sql"],
        "raw_generation": first["raw_generation"],
        "sql_error": first["sql_error"],
        "pipeline_error": "",
    }
    result.update(flat_fields)

    if not (result["ex_at_1"] <= result["oracle_at_3"] <= result["oracle_at_5"]):
        raise AssertionError("Nested Oracle@K metrics are not monotonic")
    return result


def failed_result(record, experiment_label, spec, exc):
    result = {
        "experiment": experiment_label,
        "display_name": spec["display_name"],
        "model_type": spec["model_type"],
        "position": int(record["position"]),
        "source_index": record["source_index"],
        "db_id": record["db_id"],
        "db_path": str(DB_PATHS[record["db_id"]]),
        "difficulty": record["difficulty"],
        "question": record["question"],
        "evidence": record["evidence"],
        "gold_sql": record["gold_sql"],
        "gold_valid": True,
        "gold_execution_seconds": np.nan,
        "gold_sql_error": "",
        "candidate_count": NUM_CANDIDATES,
        "unique_candidate_count": 0,
        "candidate_diversity_rate": 0.0,
        "sample_seed": int(SEED + int(record["position"])),
        "ex_at_1": False,
        "oracle_at_3": False,
        "oracle_at_5": False,
        "oracle_hit_rank": np.nan,
        "candidate_valid_rate": 0.0,
        "input_token_count": np.nan,
        "output_token_count": np.nan,
        "total_output_token_count": np.nan,
        "generation_seconds": np.nan,
        "valid_sql": False,
        "execution_match": False,
        "normalized_exact_match": False,
        "predicted_sql": "",
        "raw_generation": "",
        "sql_error": "",
        "pipeline_error": repr(exc),
    }
    for rank in range(1, NUM_CANDIDATES + 1):
        prefix = f"candidate_{rank}"
        result.update({
            f"{prefix}_raw_generation": "",
            f"{prefix}_predicted_sql": "",
            f"{prefix}_output_token_count": np.nan,
            f"{prefix}_valid_sql": False,
            f"{prefix}_execution_match": False,
            f"{prefix}_normalized_exact_match": False,
            f"{prefix}_predicted_execution_seconds": np.nan,
            f"{prefix}_sql_error": "",
        })
    return result


def evaluate_model(experiment_label, spec, model, tokenizer):
    csv_path = RUN_DIR / f"{experiment_label}_results.csv"
    config_path = RUN_DIR / f"{experiment_label}_run_config.json"
    current_config = run_config(experiment_label, spec)

    if FORCE_RESTART:
        csv_path.unlink(missing_ok=True)
        config_path.unlink(missing_ok=True)

    if config_path.exists():
        old_config = json.loads(config_path.read_text(encoding="utf-8"))
        if old_config.get("fingerprint") != current_config["fingerprint"]:
            raise RuntimeError(
                f"Previous files for {experiment_label} use a different configuration. "
                "If this change is intentional, set FORCE_RESTART=True for one run."
            )
    elif csv_path.exists() and RESUME:
        raise RuntimeError(
            f"Found {csv_path.name} without its run_config. "
            "Inspect it before intentionally setting FORCE_RESTART=True."
        )
    else:
        atomic_write_json(current_config, config_path)

    if RESUME and csv_path.exists():
        results = pd.read_csv(csv_path).to_dict("records")
        done_positions = {int(row["position"]) for row in results}
        print(f"{experiment_label}: resuming after {len(done_positions)} examples")
    else:
        results = []
        done_positions = set()

    working_df = eval_df if DEBUG_LIMIT is None else eval_df.head(int(DEBUG_LIMIT))
    pending = [
        row.to_dict()
        for _, row in working_df.iterrows()
        if int(row["position"]) not in done_positions
    ]

    started = time.perf_counter()
    for step, record in enumerate(
        tqdm(pending, desc=f"Evaluating {experiment_label}"), start=1
    ):
        try:
            generation = generate_candidates(model, tokenizer, record)
            row_result = evaluate_candidate_set(
                record, generation, experiment_label, spec
            )
        except Exception as exc:
            row_result = failed_result(record, experiment_label, spec, exc)

        results.append(row_result)
        if step % SAVE_EVERY == 0 or step == len(pending):
            result_df = pd.DataFrame(results).sort_values("position").reset_index(drop=True)
            atomic_write_csv(result_df, csv_path)
            ex1 = bool_series(result_df["ex_at_1"]).mean()
            oracle3 = bool_series(result_df["oracle_at_3"]).mean()
            oracle5 = bool_series(result_df["oracle_at_5"]).mean()
            elapsed = (time.perf_counter() - started) / 60
            print(
                f"{experiment_label}: saved={len(result_df)}, "
                f"EX@1={ex1:.2%}, Oracle@3={oracle3:.2%}, "
                f"Oracle@5={oracle5:.2%}, elapsed={elapsed:.1f} min"
            )

    result_df = pd.DataFrame(results).sort_values("position").reset_index(drop=True)
    atomic_write_csv(result_df, csv_path)
    return result_df


## 11. Run Base, original SFT, and targeted SFT sequentially

Run this cell after the five-example smoke test configuration is correct. A disconnect is safe: rerun the setup cells and this cell with `FORCE_RESTART=False` to resume.


In [31]:
!pip uninstall -y torchao

In [ ]:
RESULTS_BY_EXPERIMENT = {}

for experiment_label, spec in EXPERIMENTS.items():
    print("\n" + "=" * 80)
    print("Starting:", spec["display_name"])
    print("=" * 80)

    tokenizer = load_tokenizer()
    model = load_eval_model(spec["adapter_dir"])
    RESULTS_BY_EXPERIMENT[experiment_label] = evaluate_model(
        experiment_label, spec, model, tokenizer
    )
    display(RESULTS_BY_EXPERIMENT[experiment_label].tail(2))

    del model, tokenizer
    clear_gpu_memory()

print("Completed:", list(RESULTS_BY_EXPERIMENT))



Starting: Base + evidence


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Evaluating base_with_evidence:   0%|          | 0/500 [00:00<?, ?it/s]

base_with_evidence: saved=5, EX@1=20.00%, Oracle@3=20.00%, Oracle@5=60.00%, elapsed=0.4 min
base_with_evidence: saved=10, EX@1=30.00%, Oracle@3=30.00%, Oracle@5=50.00%, elapsed=1.1 min
base_with_evidence: saved=15, EX@1=33.33%, Oracle@3=33.33%, Oracle@5=53.33%, elapsed=1.4 min
base_with_evidence: saved=20, EX@1=45.00%, Oracle@3=45.00%, Oracle@5=60.00%, elapsed=1.7 min
base_with_evidence: saved=25, EX@1=52.00%, Oracle@3=52.00%, Oracle@5=64.00%, elapsed=2.1 min
base_with_evidence: saved=30, EX@1=46.67%, Oracle@3=46.67%, Oracle@5=56.67%, elapsed=2.5 min
base_with_evidence: saved=35, EX@1=51.43%, Oracle@3=54.29%, Oracle@5=62.86%, elapsed=2.8 min
base_with_evidence: saved=40, EX@1=52.50%, Oracle@3=55.00%, Oracle@5=62.50%, elapsed=3.2 min
base_with_evidence: saved=45, EX@1=55.56%, Oracle@3=57.78%, Oracle@5=64.44%, elapsed=3.4 min
base_with_evidence: saved=50, EX@1=60.00%, Oracle@3=62.00%, Oracle@5=68.00%, elapsed=3.7 min
base_with_evidence: saved=55, EX@1=63.64%, Oracle@3=65.45%, Oracle@5=70

,experiment,display_name,model_type,position,source_index,db_id,db_path,difficulty,question,evidence,...,candidate_4_predicted_execution_seconds,candidate_4_sql_error,candidate_5_raw_generation,candidate_5_predicted_sql,candidate_5_output_token_count,candidate_5_valid_sql,candidate_5_execution_match,candidate_5_normalized_exact_match,candidate_5_predicted_execution_seconds,candidate_5_sql_error
498,base_with_evidence,Base + evidence,base,498,498,financial,/content/bird_minidev500_sqlite/financial/financial.sqlite,moderate,"For accounts in 1993 with statement issued after transaction, list the account ID, district name and district region.",Records about district names could be found in A2; A3 contains the information about regions. 'POPLATEK PO OBRATU' stands for issuance after transaction,...,0.000811,,"SELECT T1.account_id, T4.A2, T4.A3 FROM disp AS T1 INNER JOIN account AS T2 ON T1.account_id = T2.account_id INNER JOIN trans AS T3 ON T1.account_id = T3.ac...","SELECT T1.account_id, T4.A2, T4.A3 FROM disp AS T1 INNER JOIN account AS T2 ON T1.account_id = T2.account_id INNER JOIN trans AS T3 ON T1.account_id = T3.ac...",105,True,False,False,0.000792,
499,base_with_evidence,Base + evidence,base,499,499,financial,/content/bird_minidev500_sqlite/financial/financial.sqlite,moderate,"From Year 1995 to 2000, who are the accounts holders from 'east Bohemia'. State the account ID the frequency of statement issuance.",Accounts holder refers to the person who own this account.,...,0.000367,no such column: T3.account_id,"SELECT T2.disp_id, T1.account_id, T1.frequency FROM account AS T1 INNER JOIN disp AS T2 ON T1.account_id = T2.account_id INNER JOIN district AS T3 ON T1.dis...","SELECT T2.disp_id, T1.account_id, T1.frequency FROM account AS T1 INNER JOIN disp AS T2 ON T1.account_id = T2.account_id INNER JOIN district AS T3 ON T1.dis...",107,True,False,False,0.003606,


GPU memory allocated (GB): 14.194

Starting: Original SFT + evidence


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Evaluating original_sft_with_evidence:   0%|          | 0/500 [00:00<?, ?it/s]

original_sft_with_evidence: saved=5, EX@1=20.00%, Oracle@3=20.00%, Oracle@5=20.00%, elapsed=0.9 min
original_sft_with_evidence: saved=10, EX@1=20.00%, Oracle@3=20.00%, Oracle@5=30.00%, elapsed=2.4 min
original_sft_with_evidence: saved=15, EX@1=26.67%, Oracle@3=26.67%, Oracle@5=33.33%, elapsed=3.1 min
original_sft_with_evidence: saved=20, EX@1=40.00%, Oracle@3=40.00%, Oracle@5=45.00%, elapsed=3.8 min
original_sft_with_evidence: saved=25, EX@1=48.00%, Oracle@3=48.00%, Oracle@5=52.00%, elapsed=4.7 min
original_sft_with_evidence: saved=30, EX@1=43.33%, Oracle@3=46.67%, Oracle@5=50.00%, elapsed=5.5 min
original_sft_with_evidence: saved=35, EX@1=45.71%, Oracle@3=54.29%, Oracle@5=57.14%, elapsed=6.2 min
original_sft_with_evidence: saved=40, EX@1=47.50%, Oracle@3=55.00%, Oracle@5=57.50%, elapsed=7.0 min
original_sft_with_evidence: saved=45, EX@1=51.11%, Oracle@3=57.78%, Oracle@5=60.00%, elapsed=7.5 min
original_sft_with_evidence: saved=50, EX@1=56.00%, Oracle@3=62.00%, Oracle@5=64.00%, elapsed

## 12. Summarize the three conditions


In [ ]:
EVALUATION_METRICS = {
    "ex_at_1": "EX@1",
    "oracle_at_3": "Oracle@3",
    "oracle_at_5": "Oracle@5",
}

expected_size = int(DEBUG_LIMIT) if DEBUG_LIMIT is not None else 500
required_columns = set(EVALUATION_METRICS) | {
    f"candidate_{rank}_predicted_sql"
    for rank in range(1, NUM_CANDIDATES + 1)
}

for label, df in RESULTS_BY_EXPERIMENT.items():
    if len(df) != expected_size:
        raise RuntimeError(f"{label}: expected {expected_size} rows, found {len(df)}")
    missing = required_columns - set(df.columns)
    if missing:
        raise RuntimeError(f"{label} is missing columns: {sorted(missing)}")
    if not (
        (bool_series(df["ex_at_1"]).astype(int)
         <= bool_series(df["oracle_at_3"]).astype(int)).all()
        and
        (bool_series(df["oracle_at_3"]).astype(int)
         <= bool_series(df["oracle_at_5"]).astype(int)).all()
    ):
        raise RuntimeError(f"Non-monotonic Oracle metrics in {label}")

reference_positions = set(
    RESULTS_BY_EXPERIMENT["base_with_evidence"]["position"].astype(int)
)
for label, df in RESULTS_BY_EXPERIMENT.items():
    if set(df["position"].astype(int)) != reference_positions:
        raise RuntimeError(f"Position mismatch in {label}")


def summarize(label, df):
    spec = EXPERIMENTS[label]
    ex1 = bool_series(df["ex_at_1"]).mean()
    oracle3 = bool_series(df["oracle_at_3"]).mean()
    oracle5 = bool_series(df["oracle_at_5"]).mean()
    pipeline_errors = df["pipeline_error"].fillna("").astype(str).str.len().gt(0)
    return {
        "Experiment": label,
        "Model": spec["display_name"],
        "Examples": len(df),
        "EX@1": ex1,
        "Oracle@3": oracle3,
        "Oracle@5": oracle5,
        "Oracle gain 3-1": oracle3 - ex1,
        "Oracle gain 5-1": oracle5 - ex1,
        "Valid SQL@1": bool_series(df["valid_sql"]).mean(),
        "Valid SQL across candidates": pd.to_numeric(
            df["candidate_valid_rate"], errors="coerce"
        ).mean(),
        "Avg unique candidates": pd.to_numeric(
            df["unique_candidate_count"], errors="coerce"
        ).mean(),
        "Avg candidate diversity": pd.to_numeric(
            df["candidate_diversity_rate"], errors="coerce"
        ).mean(),
        "Avg generation s/example": pd.to_numeric(
            df["generation_seconds"], errors="coerce"
        ).mean(),
        "Pipeline errors": int(pipeline_errors.sum()),
    }


summary_table = pd.DataFrame([
    summarize(label, df)
    for label, df in RESULTS_BY_EXPERIMENT.items()
])
atomic_write_csv(summary_table, RUN_DIR / "three_way_metric_summary.csv")

display(summary_table.style.format({
    "EX@1": "{:.2%}",
    "Oracle@3": "{:.2%}",
    "Oracle@5": "{:.2%}",
    "Oracle gain 3-1": "{:+.2%}",
    "Oracle gain 5-1": "{:+.2%}",
    "Valid SQL@1": "{:.2%}",
    "Valid SQL across candidates": "{:.2%}",
    "Avg unique candidates": "{:.2f}",
    "Avg candidate diversity": "{:.2%}",
    "Avg generation s/example": "{:.3f}",
}))


## 13. Paired model comparisons and statistical checks


In [ ]:
KEY_COLUMNS = [
    "position", "source_index", "db_id", "difficulty",
    "question", "evidence", "gold_sql",
]
METRIC_COLUMNS = [
    "ex_at_1", "oracle_at_3", "oracle_at_5", "oracle_hit_rank",
    "valid_sql", "candidate_valid_rate", "unique_candidate_count",
    "candidate_diversity_rate", "generation_seconds", "predicted_sql",
    "pipeline_error",
]

three_way = None
for label, df in RESULTS_BY_EXPERIMENT.items():
    part = df[KEY_COLUMNS + METRIC_COLUMNS].copy()
    part = part.rename(columns={
        column: f"{column}_{label}" for column in METRIC_COLUMNS
    })
    if three_way is None:
        three_way = part
    else:
        three_way = three_way.merge(
            part, on=KEY_COLUMNS, how="inner", validate="one_to_one"
        )

if len(three_way) != expected_size:
    raise RuntimeError(
        f"Three-way merge produced {len(three_way)} rows; expected {expected_size}"
    )

for label in EXPERIMENTS:
    for metric in EVALUATION_METRICS:
        three_way[f"{metric}_{label}"] = bool_series(
            three_way[f"{metric}_{label}"]
        )

atomic_write_csv(three_way, RUN_DIR / "three_way_per_example.csv")

PAIR_SPECS = {
    "original_sft_minus_base": (
        "base_with_evidence", "original_sft_with_evidence"
    ),
    "targeted_sft_minus_base": (
        "base_with_evidence", "targeted_sft_with_evidence"
    ),
    "targeted_sft_minus_original_sft": (
        "original_sft_with_evidence", "targeted_sft_with_evidence"
    ),
}


def paired_summary(metric, effect, control_label, treatment_label, seed_offset):
    control = three_way[f"{metric}_{control_label}"].astype(int).to_numpy()
    treatment = three_way[f"{metric}_{treatment_label}"].astype(int).to_numpy()
    difference = treatment - control
    improved = int(((control == 0) & (treatment == 1)).sum())
    regressed = int(((control == 1) & (treatment == 0)).sum())
    discordant = improved + regressed

    rng = np.random.default_rng(SEED + seed_offset)
    bootstrap_means = np.array([
        rng.choice(difference, size=len(difference), replace=True).mean()
        for _ in range(10_000)
    ])
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])

    return {
        "metric": EVALUATION_METRICS[metric],
        "effect": effect,
        "control": control_label,
        "treatment": treatment_label,
        "control_score": float(control.mean()),
        "treatment_score": float(treatment.mean()),
        "absolute_delta": float(difference.mean()),
        "improved": improved,
        "regressed": regressed,
        "net_improvement": improved - regressed,
        "bootstrap_95_ci_low": float(ci_low),
        "bootstrap_95_ci_high": float(ci_high),
        "mcnemar_exact_p_value": float(
            binomtest(improved, discordant, p=0.5).pvalue
            if discordant else 1.0
        ),
    }


paired_rows = []
seed_offset = 0
for metric in EVALUATION_METRICS:
    for effect, (control, treatment) in PAIR_SPECS.items():
        paired_rows.append(
            paired_summary(metric, effect, control, treatment, seed_offset)
        )
        seed_offset += 1

paired_effects = pd.DataFrame(paired_rows)
atomic_write_csv(paired_effects, RUN_DIR / "paired_model_effects.csv")

display(paired_effects.style.format({
    "control_score": "{:.2%}",
    "treatment_score": "{:.2%}",
    "absolute_delta": "{:+.2%}",
    "bootstrap_95_ci_low": "{:+.2%}",
    "bootstrap_95_ci_high": "{:+.2%}",
    "mcnemar_exact_p_value": "{:.4g}",
}))


## 14. Breakdown by difficulty and database


In [ ]:
def grouped_breakdown(group_column):
    rows = []
    for group_name, group in three_way.groupby(group_column, dropna=False):
        for label, spec in EXPERIMENTS.items():
            ex1 = group[f"ex_at_1_{label}"].mean()
            oracle3 = group[f"oracle_at_3_{label}"].mean()
            oracle5 = group[f"oracle_at_5_{label}"].mean()
            rows.append({
                group_column: group_name,
                "count": len(group),
                "experiment": label,
                "model": spec["display_name"],
                "EX@1": ex1,
                "Oracle@3": oracle3,
                "Oracle@5": oracle5,
                "Oracle gain 3-1": oracle3 - ex1,
                "Oracle gain 5-1": oracle5 - ex1,
                "Avg candidate diversity": pd.to_numeric(
                    group[f"candidate_diversity_rate_{label}"], errors="coerce"
                ).mean(),
            })
    return pd.DataFrame(rows)


difficulty_comparison = grouped_breakdown("difficulty")
database_comparison = grouped_breakdown("db_id")

atomic_write_csv(
    difficulty_comparison, RUN_DIR / "three_way_by_difficulty.csv"
)
atomic_write_csv(
    database_comparison, RUN_DIR / "three_way_by_database.csv"
)

breakdown_format = {
    "EX@1": "{:.2%}",
    "Oracle@3": "{:.2%}",
    "Oracle@5": "{:.2%}",
    "Oracle gain 3-1": "{:+.2%}",
    "Oracle gain 5-1": "{:+.2%}",
    "Avg candidate diversity": "{:.2%}",
}
display(difficulty_comparison.style.format(breakdown_format))
display(database_comparison.style.format(breakdown_format))


## 15. Visualize accuracy, recoverable gap, and diversity


In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))

metric_long = summary_table.melt(
    id_vars=["Experiment", "Model"],
    value_vars=["EX@1", "Oracle@3", "Oracle@5"],
    var_name="Metric",
    value_name="Score",
)
sns.barplot(
    data=metric_long,
    x="Model",
    y="Score",
    hue="Metric",
    palette="Set2",
    ax=axes[0],
)
axes[0].set_title("Overall execution accuracy")
axes[0].tick_params(axis="x", rotation=18)

sns.barplot(
    data=difficulty_comparison,
    x="difficulty",
    y="Oracle gain 5-1",
    hue="model",
    palette="Set2",
    ax=axes[1],
)
axes[1].set_title("Recoverable Oracle@5 gap")
axes[1].set_ylabel("Oracle@5 - EX@1")

sns.barplot(
    data=summary_table,
    x="Model",
    y="Avg candidate diversity",
    palette="Set2",
    ax=axes[2],
)
axes[2].set_ylim(0, 1)
axes[2].set_title("Candidate diversity")
axes[2].tick_params(axis="x", rotation=18)

plt.tight_layout()
chart_path = RUN_DIR / "three_way_oracle_diagnosis.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", chart_path)


## 16. Automatic diagnosis and next-route recommendation

Interpretation thresholds are deliberately coarse:

- Oracle@5 − EX@1 ≥ 8 points: strong selection/ranking bottleneck.
- 3–8 points: mixed generation and selection bottleneck.
- < 3 points: generation/reasoning bottleneck.

Use the full 500-example run—not the smoke test—for the final decision.


In [ ]:
def diagnose_row(row):
    gap = float(row["Oracle gain 5-1"])
    diversity = float(row["Avg candidate diversity"])
    if gap >= 0.08:
        diagnosis = "selection/ranking bottleneck"
        next_step = "build an execution-aware candidate reranker/selector"
    elif gap >= 0.03:
        diagnosis = "mixed generation and selection bottleneck"
        next_step = "combine structured planning with a lightweight selector"
    else:
        diagnosis = "generation/reasoning bottleneck"
        next_step = "test structured planning, then reasoning-aware SFT if planning helps"

    if diversity < 0.40:
        next_step += "; also improve decoding diversity before training a selector"

    return diagnosis, next_step


diagnosis_rows = []
for _, row in summary_table.iterrows():
    diagnosis, next_step = diagnose_row(row)
    diagnosis_rows.append({
        "model": row["Model"],
        "ex_at_1": row["EX@1"],
        "oracle_at_5": row["Oracle@5"],
        "oracle_gap_5_vs_1": row["Oracle gain 5-1"],
        "candidate_diversity": row["Avg candidate diversity"],
        "diagnosis": diagnosis,
        "recommended_next_step": next_step,
    })

diagnosis_table = pd.DataFrame(diagnosis_rows)

base_oracle5 = float(
    summary_table.loc[
        summary_table["Experiment"] == "base_with_evidence", "Oracle@5"
    ].iloc[0]
)
original_oracle5 = float(
    summary_table.loc[
        summary_table["Experiment"] == "original_sft_with_evidence", "Oracle@5"
    ].iloc[0]
)
targeted_oracle5 = float(
    summary_table.loc[
        summary_table["Experiment"] == "targeted_sft_with_evidence", "Oracle@5"
    ].iloc[0]
)

if base_oracle5 >= max(original_oracle5, targeted_oracle5):
    global_recommendation = (
        "Use Base + evidence as the candidate generator. "
        "Do not continue tuning the current SQL-only adapters until the selector/planning diagnosis is completed."
    )
elif targeted_oracle5 > base_oracle5:
    global_recommendation = (
        "Targeted SFT expands the reachable correct-candidate set. "
        "Keep it as a generator candidate, then determine whether its Oracle gain can be converted by a selector."
    )
else:
    global_recommendation = (
        "Original SFT has the strongest reachable candidate set. "
        "Use it as the generator candidate and test selection before further fine-tuning."
    )

decision_report = {
    "debug_limit": DEBUG_LIMIT,
    "warning": (
        "Smoke-test result only; do not make a project decision."
        if DEBUG_LIMIT is not None else None
    ),
    "per_model": diagnosis_rows,
    "global_recommendation": global_recommendation,
}
atomic_write_json(decision_report, RUN_DIR / "oracle_diagnosis_decision.json")

display(diagnosis_table.style.format({
    "ex_at_1": "{:.2%}",
    "oracle_at_5": "{:.2%}",
    "oracle_gap_5_vs_1": "{:+.2%}",
    "candidate_diversity": "{:.2%}",
}))
print("\nGLOBAL RECOMMENDATION")
print(global_recommendation)
if DEBUG_LIMIT is not None:
    print("\nWARNING: this is a smoke test. Rerun all 500 examples before deciding.")


## 17. Output files

The result directory contains:

- One resumable per-example CSV and one run-config JSON for each model.
- `three_way_metric_summary.csv`
- `three_way_per_example.csv`
- `paired_model_effects.csv`
- `three_way_by_difficulty.csv`
- `three_way_by_database.csv`
- `three_way_oracle_diagnosis.png`
- `oracle_diagnosis_decision.json`

Keep the three run-config JSON files with the CSV results. They record the adapter identity, dataset hash, prompt, and decoding settings needed to interpret or reproduce the experiment.
